# AudioSR T4 / TensorRT — изолированная проверка

Маленький блокнот только для проверки ускорения AudioSR на NVIDIA T4. Он не меняет рабочий Gradio и не вливает ничего в `main`.

**Порядок:** включить T4 → нажать «Выполнить всё» → выбрать один аудиофайл, когда появится запрос → после выбора обработка запускается сразу → получить сравнение PyTorch/TensorRT, VRAM, SNR и WAV-файлы.


In [ ]:
import platform
import subprocess
import sys

import torch

print('Python:', sys.version.split()[0])
print('Ubuntu:', platform.platform())
print('PyTorch:', torch.__version__)
print('PyTorch CUDA:', torch.version.cuda)
print('CUDA доступна:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'Нужен Colab GPU runtime: Runtime → Change runtime type → T4 GPU.'
    )
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
if 'T4' not in gpu:
    print(
        'Предупреждение: probe рассчитан прежде всего на T4; '
        'сейчас выдана другая GPU.'
    )
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path('/content/audio-restoration-colab')
BRANCH = 'agent/audiosr-t4-tensorrt'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/egor125552/audio-restoration-colab.git', str(REPO)
], check=True)
print('Код готов:', REPO)


In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '-q', 'uv==0.12.0'
], check=True)
CACHE = '/content/audio-restoration-models'
subprocess.run([
    'bash', str(REPO / 'scripts/prepare_audiosr_t4.sh'), CACHE
], check=True)
PROBE_PYTHON = str(Path(CACHE) / 'envs/audiosr_trt/bin/python')
print('TensorRT-среда готова:', PROBE_PYTHON)


In [ ]:
import subprocess
from pathlib import Path

from google.colab import files

print('Выбери один аудиофайл. После выбора обработка начнётся сразу.', flush=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Файл не выбран.')

input_name, input_bytes = next(iter(uploaded.items()))
input_file = Path('/content') / input_name
if not input_file.is_file():
    input_file.write_bytes(input_bytes)

print(f'Файл принят: {input_file.name}', flush=True)
print('Запускаю probe на первых 5.12 секунды аудио…', flush=True)

results_dir = Path('/content/audiosr-t4-results')
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / 'probe.log'
command = [
    PROBE_PYTHON,
    str(REPO / 'scripts/probe_audiosr_t4.py'),
    '--input', str(input_file),
    '--output-dir', str(results_dir),
    '--mode', 'basic',
    '--steps', '30',
    '--guidance', '3.5',
    '--seed', '42',
    '--runs', '2',
]

process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
last_lines = []
with log_path.open('w', encoding='utf-8') as log_file:
    if process.stdout is None:
        raise RuntimeError('Не удалось подключить вывод probe.')
    for line in process.stdout:
        print(line, end='', flush=True)
        log_file.write(line)
        log_file.flush()
        last_lines.append(line.rstrip())
        if len(last_lines) > 80:
            last_lines.pop(0)

probe_return_code = process.wait()
if probe_return_code == 0:
    print('Probe завершён успешно. Перехожу к результатам.', flush=True)
else:
    print('', flush=True)
    print(f'PROBE УПАЛ: код {probe_return_code}.', flush=True)
    print(f'Полный лог сохранён: {log_path}', flush=True)
    print('Последние строки лога:', flush=True)
    for line in last_lines[-40:]:
        print(line, flush=True)


In [ ]:
from pathlib import Path

results_dir = Path('/content/audiosr-t4-results')
log_path = results_dir / 'probe.log'
wav_files = sorted(results_dir.glob('*.wav'))

if wav_files:
    print(f'Готово: найдено WAV-файлов: {len(wav_files)}')
    for path in wav_files:
        print('-', path.name)
else:
    print('WAV-файлов пока нет: probe завершился с ошибкой.')
    if log_path.is_file():
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('Последние строки probe.log:')
        for line in lines[-40:]:
            print(line)
    else:
        print('probe.log тоже не найден; ошибка произошла до запуска probe.')


In [ ]:
from pathlib import Path

from IPython.display import Audio, display

results_dir = Path('/content/audiosr-t4-results')
wav_files = sorted(results_dir.glob('*.wav'))

if not wav_files:
    print('WAV-файлов пока нет. Сначала должна успешно завершиться ячейка с probe.')
else:
    print('WAV для сравнения:')
    for path in wav_files:
        print('-', path.name)
        display(Audio(filename=str(path)))
